# Neural Amp Modeler — Temporal effects (CUDA / Colab / Kaggle)

**Before you run:** enable a **GPU** (Colab: *Runtime → Change runtime type → GPU*. Kaggle: *Settings → Accelerator → GPU*).

This notebook trains the **temporal** (delay/reverb-style) model from this fork. It does **not** use the standard Colab trainer `nam.train.colab.run()` — that path targets the classic WaveNet-style amp model only.

Exports are still plugin-compatible **`.nam`** files from the same export path as local `train.py`.

## Step 1: Get data

* **Dry signal:** `input.wav` (the same reamp / DI file you used for the wet capture).
* **Wet capture:** `output.wav` (reamp through your time-based effect). Prefer **48 kHz**, **24-bit**, **mono** (same as the [official NAM Colab tutorial](https://neural-amp-modeler.readthedocs.io/en/stable/tutorials/colab.html)). This fork also supports **stereo** pairs if channel counts match.
* **Upload:** Colab — use the folder panel and upload both files (often under `/content`). Kaggle — add a **Dataset** containing `input.wav` / `output.wav` and reference paths under `/kaggle/input/...`, or copy them into `/kaggle/working`.

Set `INPUT_WAV` and `OUTPUT_WAV` in the next code section to match where those files landed.

### Faster runs (optional)

* **`steps`** — wall time scales about linearly with `max_steps`.
* **`precision`** — use **`16-mixed`** on CUDA for a large throughput win; keep **`32-true`** if you hit instability.
* **`mrstft_weight`** — set to **`0`** to skip MR-STFT in the *training* loss for quick experiments (often much faster; quality may suffer).
* **`val_check_interval`** — validation is expensive, but PyTorch Lightning requires this to be **≤ train batches per epoch** (≈ `ceil(epoch_steps / batch_size)`). Example: `epoch_steps=2000`, `batch_size=8` → at most **250**. Larger values need a higher `epoch_steps` (e.g. `8000`+ for interval `1000`).
* **`preview_every` / `checkpoint_every`** — larger values mean less disk I/O (preview WAVs and checkpoints).
* **`batch_size`** — increase while VRAM allows; stereo runs effectively double batch width after channel flattening.
* **`context` / `target`** — shorter windows speed up each step but cap how much reverb tail the model sees at once (quality tradeoff).
* **`hidden_size` / `local_layers`** — smaller / fewer layers train faster with less capacity.

`epoch_steps` changes how many batches define one DataLoader epoch (and scales default validation length). It does **not** replace lowering `steps` if you want fewer total optimizer updates.

### Kaggle notes

* Write outputs under **`/kaggle/working`** so they persist for download.
* Put training WAVs in an **Input Dataset** and point `INPUT_WAV` / `OUTPUT_WAV` at `/kaggle/input/<dataset>/...`.
* If DataLoader workers crash, set **`num_workers`** to **`0`** or **`2`**.
* Some Kaggle sessions provide **Tesla P100 (SM 6.0)**. Newer PyTorch/cuDNN builds may not support SM < 7.5. Use install-cell `TORCH_PROFILE="legacy_sm60"`, then **restart runtime** before training. For fastest GPU training, prefer T4/L4/A10 runtimes when available.

In [ ]:
# Install this fork (safe for Colab/Kaggle shared environments).
# Uses --no-deps by default to avoid breaking preinstalled runtime packages.

import importlib.util
from pathlib import Path
import re
import subprocess
import sys


def _pip_install(*args: str) -> None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


def _to_git_url(repo_url: str, branch: str) -> str:
    url = repo_url.strip().rstrip("/")
    if not url.startswith("git+"):
        url = "git+" + url
    if "@" not in url.split("#", 1)[0]:
        url = f"{url}@{branch}"
    return url


def _read_model_version_without_import() -> str:
    spec = importlib.util.find_spec("nam")
    if spec is None or spec.origin is None:
        return "unknown"
    nam_init = Path(spec.origin)
    constants_path = nam_init.parent / "models" / "_constants.py"
    if not constants_path.exists():
        return "unknown"
    text = constants_path.read_text()
    m = re.search(r'MODEL_VERSION\s*=\s*"([^"]+)"', text)
    return m.group(1) if m else "unknown"


INSTALL_MODE = "git"  #@param ["git", "local_editable"] {type:"string"}
# Full git HTTPS URL to the repo root (must contain this fork's pyproject.toml), e.g. https://github.com/you/neural-amp-modeler.git
GIT_REPO_URL = "https://github.com/robjlyons/nam-time-temporal.git"  #@param {type:"string"}
GIT_BRANCH = "main"  #@param {type:"string"}
# After unzipping or cloning the repo into the runtime, point here for editable install:
LOCAL_PACKAGE_DIR = "/content/neural-amp-modeler"  #@param {type:"string"}
FORCE_REINSTALL = True  #@param {type:"boolean"}
INSTALL_DEPS = False  #@param {type:"boolean"}
REPAIR_TORCHVISION = True  #@param {type:"boolean"}
INSTALL_TORCH_STACK = True  #@param {type:"boolean"}
TORCH_PROFILE = "auto"  #@param ["auto", "legacy_sm60", "modern"] {type:"string"}


def _torch_stack_health() -> tuple[bool, str]:
    try:
        import torch  # noqa: F401
    except Exception as e:
        return False, f"torch import failed: {e}"
    try:
        tv_spec = importlib.util.find_spec("torchvision")
        if tv_spec is None:
            return True, "torchvision not installed (OK for training)."
        import torchvision  # noqa: F401
        return True, "torchvision import OK."
    except Exception as e:
        return False, f"torchvision import failed: {e}"


def _detect_legacy_sm60() -> bool:
    try:
        out = subprocess.check_output(
            ["nvidia-smi", "--query-gpu=compute_cap", "--format=csv,noheader"],
            text=True,
        ).strip().splitlines()
        for raw in out:
            cc = float(raw.strip())
            if cc < 7.5:
                return True
    except Exception:
        # If detection fails, keep default profile behavior.
        return False
    return False


def _install_torch_stack(profile: str) -> None:
    if profile == "legacy_sm60":
        # CUDA 12.1 wheels from PyTorch 2.4.x still support Pascal-class GPUs (e.g. P100, SM 6.0).
        _pip_install(
            "--upgrade",
            "--force-reinstall",
            "--no-cache-dir",
            "--index-url",
            "https://download.pytorch.org/whl/cu121",
            "torch==2.4.1",
            "torchvision==0.19.1",
            "torchaudio==2.4.1",
        )
    elif profile == "modern":
        # Keep runtime defaults for modern GPUs unless user explicitly wants to pin.
        print("Using runtime-provided torch stack for modern GPU profile.")
    else:
        raise ValueError(f"Unsupported TORCH_PROFILE={profile}")


# Only check for top-level `nam` package here. Probing `nam.training` can import
# heavy deps (lightning/torchmetrics/torchvision) and fail in partially-broken runtimes.
need_install = (
    FORCE_REINSTALL
    or importlib.util.find_spec("nam") is None
)

resolved_profile = TORCH_PROFILE
if resolved_profile == "auto":
    resolved_profile = "legacy_sm60" if _detect_legacy_sm60() else "modern"
print("Resolved torch profile:", resolved_profile)

if INSTALL_TORCH_STACK:
    _install_torch_stack(resolved_profile)
    if resolved_profile == "legacy_sm60":
        print("[important] Legacy SM60 torch stack installed. Restart runtime before running training cell.")

if need_install:
    install_args = ["--upgrade"]
    if FORCE_REINSTALL:
        install_args += ["--force-reinstall", "--no-cache-dir"]
    if not INSTALL_DEPS:
        install_args += ["--no-deps"]

    if INSTALL_MODE == "git":
        package_ref = _to_git_url(GIT_REPO_URL, GIT_BRANCH)
        print("Installing from", package_ref)
        _pip_install(*install_args, package_ref)
    else:
        print("Editable install from", LOCAL_PACKAGE_DIR)
        _pip_install(*install_args, "-e", LOCAL_PACKAGE_DIR)
else:
    print("nam already importable; skipping install.")

ok, msg = _torch_stack_health()
print("Torch stack check:", msg)
if (not ok) and REPAIR_TORCHVISION:
    print("Repairing broken torchvision installation (uninstalling torchvision)...")
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "torchvision"])
    ok2, msg2 = _torch_stack_health()
    print("Torch stack check after repair:", msg2)
    if not ok2:
        raise RuntimeError(
            "Torch stack is still broken. Restart runtime, then run install cell again."
        )

# Verify active model export version without importing nam.models (avoids scipy import side-effects).
print("Active NAM model export version:", _read_model_version_without_import())

# If runtime imports are still stale after install, do Runtime -> Restart session and re-run.
print("Install cell done.")

In [ ]:
from datetime import datetime, timezone
import json
from pathlib import Path
import shutil
import torch

try:
    from nam.training.config import TemporalTrainingConfig
    from nam.training.engine import train_temporal_model
except Exception as e:
    raise RuntimeError(
        "Failed to import NAM training modules. Re-run the install cell, then restart runtime and try again. "
        f"Original error: {e}"
    )

# --- Paths (edit for your environment) ---
INPUT_WAV = "/content/input.wav"  #@param {type:"string"}
OUTPUT_WAV = "/content/output.wav"  #@param {type:"string"}
OUTDIR = "/content/nam_temporal_out"  #@param {type:"string"}
RESUME_CKPT = ""  #@param {type:"string"}

# --- Training ---
steps = 20000  #@param {type:"integer"}
batch_size = 8  #@param {type:"integer"}
context = 8192  #@param {type:"integer"}
target = 8192  #@param {type:"integer"}
overlap = 1024  #@param {type:"integer"}
epoch_steps = 2000  #@param {type:"integer"}
hidden_size = 48  #@param {type:"integer"}
train_burn_in = 0  #@param {type:"integer"}
train_truncate = 0  #@param {type:"integer"}
local_layers = 2  #@param {type:"integer"}
learning_rate = 3e-4  #@param {type:"number"}
esr_denominator_floor = 1e-8  #@param {type:"number"}
esr_weight = 0.25  #@param {type:"number"}
active_sampling_ratio = 0.7  #@param {type:"number"}
active_rms_quantile = 0.8  #@param {type:"number"}
active_window_min_rms = 0.0  #@param {type:"number"}
validation_require_active = True  #@param {type:"boolean"}
lr_scheduler = "reduce_on_plateau"  #@param ["none", "reduce_on_plateau"] {type:"string"}
lr_factor = 0.7  #@param {type:"number"}
lr_patience = 2  #@param {type:"integer"}
lr_min = 1e-6  #@param {type:"number"}
val_check_interval = 250  #@param {type:"integer"}
checkpoint_every = 500  #@param {type:"integer"}
preview_every = 1000  #@param {type:"integer"}
log_every = 50  #@param {type:"integer"}
num_workers = 2  #@param {type:"integer"}
prefetch_factor = 2  #@param {type:"integer"}
precision = "16-mixed"  #@param ["32-true", "16-mixed"] {type:"string"}
mrstft_weight = 0.0002  #@param {type:"number"}
device = "auto"  #@param ["auto", "cpu", "gpu"] {type:"string"}
force_mono = False  #@param {type:"boolean"}
alignment_mode = "global"  #@param ["none", "global", "piecewise"] {type:"string"}
piecewise_block_samples = 65536  #@param {type:"integer"}
piecewise_hop_samples = 0  #@param {type:"integer"}
piecewise_smooth_blocks = 3  #@param {type:"integer"}
piecewise_max_residual_delay_samples = 512  #@param {type:"integer"}
piecewise_min_peak_ratio = 1.02  #@param {type:"number"}
normalization_mode = "none"  #@param ["none", "rms_match", "affine"] {type:"string"}
remove_dc = False  #@param {type:"boolean"}
min_alignment_peak_ratio = 1.25  #@param {type:"number"}
max_residual_delay_std_samples = 4.0  #@param {type:"number"}
clip_threshold = 0.999  #@param {type:"number"}
max_clip_fraction = 0.02  #@param {type:"number"}
fail_on_quality_gates = False  #@param {type:"boolean"}
enable_logger = False  #@param {type:"boolean"}
use_tensorboard = False  #@param {type:"boolean"}

outdir = Path(OUTDIR)
outdir.mkdir(parents=True, exist_ok=True)

# Guard against stale exports from failed/older runs.
existing_model_path = outdir / "model.nam"
if existing_model_path.exists():
    ts = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    archived = outdir / f"model.preexisting.{ts}.nam"
    shutil.move(str(existing_model_path), str(archived))
    print(f"[warn] Archived preexisting model export to: {archived}")

if use_tensorboard:
    from IPython import get_ipython

    # TensorBoard can fail in some Kaggle images due to protobuf/tensorboard conflicts.
    # Keep training going even if visualization setup fails.
    _ipy = get_ipython()
    if _ipy is not None:
        try:
            _ipy.run_line_magic("load_ext", "tensorboard")
            _ipy.run_line_magic("tensorboard", f"--logdir {outdir}")
        except Exception as e:
            print(f"[warn] TensorBoard setup skipped: {e}")

resolved_device = str(device)
resolved_precision = str(precision)
resolved_profile = str(globals().get("resolved_profile", "modern"))

if resolved_device in {"auto", "gpu"} and torch.cuda.is_available():
    cc = torch.cuda.get_device_capability(0)
    if cc < (7, 5) and resolved_profile != "legacy_sm60":
        gpu_name = torch.cuda.get_device_name(0)
        raise RuntimeError(
            f"CUDA device {gpu_name} has capability sm_{cc[0]}{cc[1]} (< sm_75), but "
            "legacy torch profile is not active. Re-run install cell with "
            "TORCH_PROFILE='legacy_sm60', then restart runtime before training."
        )

if resolved_device == "cpu" and resolved_precision == "16-mixed":
    print("[warn] 16-mixed is not supported on CPU; using 32-true.")
    resolved_precision = "32-true"

print(f"Resolved training device={resolved_device} precision={resolved_precision}")

cfg = TemporalTrainingConfig(
    input_wav=Path(INPUT_WAV),
    output_wav=Path(OUTPUT_WAV),
    outdir=outdir,
    max_steps=int(steps),
    batch_size=int(batch_size),
    num_workers=int(num_workers),
    persistent_workers=int(num_workers) > 0,
    prefetch_factor=int(prefetch_factor),
    context_samples=int(context),
    target_samples=int(target),
    overlap_samples=int(overlap),
    epoch_steps=int(epoch_steps),
    hidden_size=int(hidden_size),
    train_burn_in=int(train_burn_in) if int(train_burn_in) > 0 else None,
    train_truncate=int(train_truncate) if int(train_truncate) > 0 else None,
    local_layers=int(local_layers),
    learning_rate=float(learning_rate),
    lr_scheduler=None if str(lr_scheduler) == "none" else str(lr_scheduler),
    lr_factor=float(lr_factor),
    lr_patience=int(lr_patience),
    lr_min=float(lr_min),
    esr_denominator_floor=float(esr_denominator_floor) if float(esr_denominator_floor) > 0 else None,
    esr_weight=float(esr_weight) if float(esr_weight) > 0 else None,
    active_sampling_ratio=max(0.0, min(1.0, float(active_sampling_ratio))),
    active_rms_quantile=max(0.0, min(1.0, float(active_rms_quantile))),
    active_window_min_rms=float(active_window_min_rms) if float(active_window_min_rms) > 0 else None,
    validation_require_active=bool(validation_require_active),
    val_check_interval=int(val_check_interval),
    checkpoint_every_n_steps=int(checkpoint_every),
    preview_every_n_steps=int(preview_every),
    log_every_n_steps=int(log_every),
    precision=resolved_precision,
    enable_logger=bool(enable_logger),
    mrstft_weight=float(mrstft_weight) if float(mrstft_weight) > 0 else None,
    resume=Path(RESUME_CKPT) if str(RESUME_CKPT).strip() else None,
    device=resolved_device,
    force_mono=bool(force_mono),
    alignment_mode=str(alignment_mode),
    piecewise_block_samples=int(piecewise_block_samples),
    piecewise_hop_samples=(int(piecewise_hop_samples) if int(piecewise_hop_samples) > 0 else None),
    piecewise_smooth_blocks=int(piecewise_smooth_blocks),
    piecewise_max_residual_delay_samples=int(piecewise_max_residual_delay_samples),
    piecewise_min_peak_ratio=float(piecewise_min_peak_ratio),
    normalization_mode=str(normalization_mode),
    remove_dc=bool(remove_dc),
    min_alignment_peak_ratio=float(min_alignment_peak_ratio),
    max_residual_delay_std_samples=float(max_residual_delay_std_samples),
    clip_threshold=float(clip_threshold),
    max_clip_fraction=float(max_clip_fraction),
    fail_on_quality_gates=bool(fail_on_quality_gates),
)

train_temporal_model(cfg)

export_path = outdir / "model.nam"
if not export_path.exists():
    raise FileNotFoundError(f"Expected export at {export_path}, but file was not created.")

with export_path.open("r") as fp:
    export_dict = json.load(fp)

export_version = str(export_dict.get("version", "missing"))
export_arch = str(export_dict.get("architecture", "missing"))
if export_version != "0.5.4":
    raise RuntimeError(
        f"Exported model version mismatch: expected 0.5.4 but got {export_version}. "
        f"Export path: {export_path}"
    )

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
plugin_copy_path = outdir / f"model_v{export_version}_{stamp}.nam"
shutil.copy2(export_path, plugin_copy_path)

print(f"Training finished. Export: {export_path}")
print(f"Verified export version={export_version} architecture={export_arch}")
print(f"Plugin-ready copy (use this file): {plugin_copy_path}")

In [ ]:
# Step 3: ESR plot (prediction vs target)
# Defaults now avoid startup/silent regions that are outside typical supervised windows.

import matplotlib.pyplot as plt
import torch

from nam.data_streaming import load_audio_pair
from nam.inference.offline import process_chunked
from nam.models import init_from_nam

PLOT_CHANNEL = 0  #@param {type:"integer"}
PLOT_START = 8191  #@param {type:"integer"}
PLOT_SAMPLES = 4096  #@param {type:"integer"}
PLOT_CHUNK_SIZE = 16384  #@param {type:"integer"}
AUTO_FIND_ACTIVE_SEGMENT = True  #@param {type:"boolean"}
ESR_DENOM_FLOOR = 1e-8  #@param {type:"number"}


def _esr_with_floor(pred: torch.Tensor, target: torch.Tensor, floor: float) -> float:
    # pred/target: (N,)
    num = torch.mean((pred - target) ** 2)
    den = torch.mean(target**2).clamp_min(float(floor))
    return float(num / den)


model_path = outdir / "model.nam"
if not model_path.exists():
    raise FileNotFoundError(f"Expected exported model at {model_path}. Run training first.")

pair = load_audio_pair(INPUT_WAV, OUTPUT_WAV, target_sample_rate=None, force_mono=bool(force_mono))

with model_path.open("r") as fp:
    model = init_from_nam(__import__("json").load(fp))
model.eval()

x = torch.from_numpy(pair.dry).float()
y = torch.from_numpy(pair.wet).float()

c = max(0, min(int(PLOT_CHANNEL), x.shape[0] - 1))
y_hat = process_chunked(model, x[c], chunk_size=int(PLOT_CHUNK_SIZE)).detach().cpu()
y_ref = y[c, : y_hat.shape[-1]].detach().cpu()

full_esr = _esr_with_floor(y_hat, y_ref, ESR_DENOM_FLOOR)

plot_len = max(1, int(PLOT_SAMPLES))
default_start = max(0, min(int(PLOT_START), y_hat.shape[-1] - 1))

if AUTO_FIND_ACTIVE_SEGMENT and y_ref.numel() > plot_len:
    # Search for a high-energy target region in the supervised area first.
    step = max(1, plot_len // 4)
    sup_start = max(0, int(context) - 1)
    max_start = y_ref.shape[-1] - plot_len
    if max_start < sup_start:
        sup_start = 0
    search = y_ref[sup_start : max_start + plot_len]
    if search.numel() >= plot_len:
        windows = search.unfold(0, plot_len, step)
        energies = torch.mean(windows**2, dim=1)
        best = int(torch.argmax(energies))
        start = min(sup_start + best * step, max_start)
    else:
        start = default_start
else:
    start = default_start

stop = min(y_hat.shape[-1], start + plot_len)
pred_seg = y_hat[start:stop]
target_seg = y_ref[start:stop]
segment_esr = _esr_with_floor(pred_seg, target_seg, ESR_DENOM_FLOOR)

plt.figure(figsize=(12, 4))
plt.plot(pred_seg.numpy(), label="Prediction")
plt.plot(target_seg.numpy(), "--", label="Target")
plt.title(f"Segment ESR={segment_esr:.3f} | Full ESR={full_esr:.3f}")
plt.legend()
plt.tight_layout()
plt.show()

print(
    f"Plotted channel={c}, samples=[{start}:{stop}] | "
    f"segment_esr={segment_esr:.6f} full_esr={full_esr:.6f} "
    f"(denom_floor={ESR_DENOM_FLOOR})"
)

## Step 4: Check results and download

* **Plugin-ready model** — use the timestamped file printed at end of training (`model_v0.5.4_*.nam`).
* **`model.nam`** — latest export in `OUTDIR`.
* **`checkpoints/`** — Lightning checkpoints (`step...ckpt`).
* **`previews/`** — periodic WAV comparisons when previews are enabled.
* TensorBoard — charts are logged under `OUTDIR` (versioned `lightning_logs` subfolders).

### Quick checklist before loading in plugin

* Confirm the printed export version is **`0.5.4`**.
* Confirm the file modified time matches your latest run.
* Load the printed timestamped copy, not an older downloaded file with the same name.

### Enjoy

Load the verified plugin-ready `.nam` file in Neural Amp Modeler.